# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [24]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [25]:
!git clone https://github.com/adam-malyshev/lerobot-audio.git
!conda install ffmpeg=7.1.1 -c conda-forge
!cd lerobot-audio && git checkout fix/audio_recording && pip install -e .

fatal: destination path 'lerobot-audio' already exists and is not an empty directory.
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | / - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Already on 'fix/audio_recording'
Your branch is up to date with 'origin/fix/audio_recording'.
Obtaining file:///content/lerobot-audio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.4.1-0.editable-py3-none-any.whl size=15716 sha256=523696aaddb5580c161c8dd0a194b22a1d51bd07c4f27dfe9d15580101f888ea
  Stored in directo

## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [26]:
!wandb login

/usr/local/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## Install SmolVLA dependencies

In [27]:
!cd lerobot-audio && pip install -e ".[smolvla, audio]"

Obtaining file:///content/lerobot-audio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.4.1-0.editable-py3-none-any.whl size=15716 sha256=7af9799eb1a1475abc773d56b17fcacfb2c79359608a121ce7ba99400c3df440
  Stored in directory: /tmp/pip-ephem-wheel-cache-26i7g9io/wheels/23/19/08/33520190e19bcc224c03c0475939567485e3318da98015612b
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.4.1
    Uninstalling lerobot-0.4.1:
      Successfully uninstalled lerobot-0.4.1


## Login into Hugging Face Hub
Now after training is done login into the Hugging Face hub and upload the last checkpoint

In [28]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) 
Token is valid (permission: read).
The token `mac_

## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [39]:
!rm -rf lerobot-audio/outputs

In [40]:
!cd lerobot-audio && git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 493 bytes | 246.00 KiB/s, done.
From https://github.com/adam-malyshev/lerobot-audio
   abb228b..e29a2e5  fix/audio_recording -> origin/fix/audio_recording
Updating abb228b..e29a2e5
Fast-forward
 src/lerobot/scripts/train_audio.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [41]:
!cd lerobot-audio && python src/lerobot/scripts/train_audio.py \
  --base_policy_path=lerobot/smolvla_base \
  --policy.repo_id=AdamMalyshev/audiosmolvla \
  --dataset.repo_id=AdamMalyshev/bell_ring_put_tape_in_bin \
  --batch_size=64 \
  --steps=20000 \
  --save_freq=1000 \
  --output_dir=outputs/train/audiosmolvla \
  --job_name=audiosmolvla_training \
  --policy.device=cuda \
  --wandb.enable=true

INFO 2025-12-15 01:41:16 ain_audio.py:89 {'base_policy_path': 'lerobot/smolvla_base',
 'batch_size': 64,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
             

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [44]:
!tar -czvf audiosmolvla_training_state2.tar.gz lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state

lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/rng_state.safetensors
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/scheduler_state.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/training_step.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/optimizer_param_groups.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/training_state/optimizer_state.safetensors


In [45]:
!tar -czvf audiosmolvla_pretrained_model2.tar.gz lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model

lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/policy_postprocessor_step_0_unnormalizer_processor.safetensors
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/policy_preprocessor_step_5_normalizer_processor.safetensors
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/policy_preprocessor.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/model.safetensors
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/policy_postprocessor.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/config.json
lerobot-audio/outputs/train/audiosmolvla/checkpoints/last/pretrained_model/train_config.json


In [46]:
!mv /content/audiosmolvla_training_state2.tar.gz /content/drive/MyDrive/

In [47]:
!mv /content/audiosmolvla_pretrained_model2.tar.gz /content/drive/MyDrive/